# Fine-tuning LFM2.5-350M for schema-compliant structured output with GRPO

### A hands-on, ~100-step run on a free Colab T4

This notebook reconstructs and operationalises the recipe from
**"Fine-tuning a 350M Model for Better Structured Outputs in 100 GRPO Steps"**
by Leonie Monigatti, Ben Burtenshaw and Sergio Paniego
([Hugging Face blog, 3 September 2026](https://huggingface.co/blog/grpo-with-trl-ifstruct)),
which reports **22.6% → 29.7%** on [IFStruct](https://github.com/Liquid4All/ifstruct)
for `LiquidAI/LFM2.5-350M` after roughly 500 samples and 100 GRPO steps.
A later, longer Liquid run on the same checkpoint reached **44.9%**.

**The task is deliberately narrow.** Emit JSON or YAML that matches a requested schema:
right fields, right types, right enums, right item count, right wrapper, right fencing,
no stray commentary. *Content quality is not scored at all.* That is what makes it
learnable by a 350M model in an afternoon — the reward is a program, not a judge.

---

### What you will actually build

| # | Step | Runs on |
|---|------|---------|
| 1 | Install the **official IFStruct validator** so reward == benchmark metric | CPU |
| 2 | Load LFM2.5-350M, discover its real LoRA target names, attach an adapter | GPU |
| 3 | Measure a **baseline** on the real frozen IFStruct test set | GPU |
| 4 | Generate a **500-row IFStruct-style training set** (the public split is test-only) | CPU |
| 5 | Build **7 staged reward functions** on top of the official validator | CPU |
| 6 | Run **100 GRPO steps** with TRL | GPU |
| 7 | Re-measure, break the delta down by axis, and ship an inference wrapper | GPU |

### Honest expectations before you start

* **This is a first loop, not a leaderboard run.** 100 steps × 500 prompts is a
  *cheap probe*. Expect a visible lift of a few points, high variance between seeds,
  and nothing close to 44.9% — that needs more data and 3–5× the steps (§9).
* **Your baseline number will not be exactly 22.6%.** The published figure uses
  `max_tokens=8000` against a served endpoint. On a T4 you will cap generation far
  lower and score a subset, so absolute numbers shift. **The before/after delta,
  measured under identical settings, is the thing that means something.**
* **Runtime:** ~15 min setup + baseline, ~25–40 min training, ~10 min final eval.
  Comfortably inside a free Colab session if you keep the completion cap modest.

> **Runtime → Change runtime type → T4 GPU** before running anything below.

## 1. Why GRPO, and why it works at 350M parameters

### SFT teaches tokens; GRPO teaches constraints

Supervised fine-tuning maximises the likelihood of reference tokens. It happily
teaches a model what schema-shaped text *looks like*, but it has no way to express
*"this must parse"* or *"this field must be an integer between 1 and 8"*. Those are
properties of the **whole decoded string**, checkable only after generation.

**Group Relative Policy Optimization** works exactly there. For each prompt it samples
a group of $G$ completions $o_1 \dots o_G$, scores each with a program, and turns the
group into a baseline for itself:

$$A_i = \frac{r_i - \operatorname{mean}(r_1 \dots r_G)}{\operatorname{std}(r_1 \dots r_G)}$$

Completions better than their peers get pushed up, worse ones down. **No critic network
is trained** — the group *is* the value estimate — which is precisely why this fits in a
T4's memory budget. With `beta=0.0` there is no reference model either, so you hold one
set of weights plus a LoRA adapter.

### Why a small model picks this up fast

The reward is **dense, verifiable and cheap**: a parse attempt plus a schema walk, ~1 ms
of CPU, no LLM judge, no human labels, zero API cost. And the skill being learned —
"close the brace, honour the enum, stop talking" — is a *format* skill, largely
independent of world knowledge. That is the one kind of thing a 350M model has ample
capacity for.

`LFM2.5-350M` is a good base for it: a hybrid short-convolution + grouped-query-attention
model (16 layers, 32k context) that Liquid trained for extraction, structured output and
tool use rather than open-ended reasoning.

### The failure modes we are actually training away

| Failure | What it looks like |
|---|---|
| Unparseable | truncated JSON, trailing prose inside the fence, single quotes |
| Wrong container | bare `[...]` when a `{"key": [...]}` wrapper was demanded (or vice versa) |
| Fencing | missing ```` ```json ```` fence, or a fence when raw output was demanded |
| Commentary | *"Sure! Here's your JSON:"* |
| Schema drift | invented keys, string `"3"` where an integer was required, enum near-misses |
| Miscount | 4 items when the prompt said 2 |

## 2. Environment

Two installs matter here:

* `trl` ≥ 1.5 and `transformers` ≥ 5.0 — LFM2.5 is a **native** architecture in
  transformers v5, so `trust_remote_code` is *not* required.
* **`ifstruct` straight from Liquid's repo.** This is the load-bearing choice in this
  notebook: it gives us `validate_response()`, the *exact* function that produces the
  published 22.6% / 29.7% numbers, plus the frozen 2,000-row test set. We will use it
  as both our reward and our metric, so there is no scoring drift between the two.

In [ ]:
%pip install -q -U "transformers>=5.0" "trl>=1.5" "peft>=0.17" "accelerate>=1.0" \
                   "datasets>=3.0" "bitsandbytes" "pyyaml" "matplotlib"
# The official IFStruct evaluator + frozen 2k test set (pure-python: PyYAML/requests only)
%pip install -q "git+https://github.com/Liquid4All/ifstruct.git"

In [ ]:
import json, os, random, re, time
from collections import Counter
from functools import lru_cache

import torch

print("torch      :", torch.__version__)
import transformers, trl, peft, datasets
print("transformers:", transformers.__version__)
print("trl         :", trl.__version__)
print("peft        :", peft.__version__)
print("datasets    :", datasets.__version__)

if not torch.cuda.is_available():
    raise SystemExit("No GPU visible. Runtime -> Change runtime type -> T4 GPU.")

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
# bf16 needs Ampere (SM 8.0+). A T4 is Turing (7.5) -> fp16 only. Forcing bf16 there is
# the single most common reason this recipe crawls or silently produces NaNs.
BF16_OK = CAPABILITY[0] >= 8
DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print(f"\nGPU        : {GPU_NAME} (SM {CAPABILITY[0]}.{CAPABILITY[1]}, {TOTAL_VRAM_GB:.1f} GB)")
print(f"bf16 usable: {BF16_OK}  ->  training dtype {str(DTYPE).split('.')[-1]}")

### Configuration in one place

Everything you might reasonably tune lives here. The defaults are chosen for a **free
T4**; the comments say what to change and why.

In [ ]:
MODEL_ID = "LiquidAI/LFM2.5-350M"

# --- data -------------------------------------------------------------------------
N_TRAIN            = 500     # synthetic IFStruct-style prompts (see section 6)
N_DEV              = 100     # held-out synthetic rows, for sanity checks
N_EVAL             = 150     # rows sampled from the REAL frozen IFStruct test set
EVAL_SEED          = 1234    # fixed -> before and after score the identical subset

# --- generation / eval ------------------------------------------------------------
EVAL_MAX_NEW_TOKENS = 640    # official harness uses 8000; a T4 cannot afford that.
                             # Same value before AND after, so the delta stays honest.
EVAL_BATCH_SIZE     = 8      # drop to 4 if you OOM during eval

# --- training ---------------------------------------------------------------------
MAX_STEPS           = 100    # the "100 GRPO steps" budget from the article
NUM_GENERATIONS     = 4      # group size G. 2 if you OOM; 8 for a stronger signal
PER_DEVICE_BS       = 4      # completions per micro-batch (memory knob)
GRAD_ACCUM          = 5      # 4*5 = 20 completions/step = 5 prompts/step
                             # -> 100 steps * 5 prompts = 500 prompts = one epoch
LEARNING_RATE       = 2e-5   # see the note in section 8 on LoRA vs full-FT LR
MAX_COMPLETION_LEN  = 640
LORA_R              = 16

LOAD_IN_4BIT        = False  # 350M in fp16 is ~0.7 GB: quantising costs speed and
                             # accuracy for no memory you actually need. Flip to True
                             # only if you are sharing the GPU with something else.

OUTPUT_DIR   = "lfm25-350m-ifstruct-grpo"
ADAPTER_DIR  = "lfm25-350m-struct-adapter"

random.seed(0)
torch.manual_seed(0)
print("Config loaded.")

## 3. The benchmark: what "pass" actually means

IFStruct is a 2,000-example frozen **test** set. Each row is a prompt plus the machine-
checkable spec of a correct answer. Crucially it presents those requirements the many
ways real users do: raw JSON Schema, a markdown field table, a pseudo-YAML skeleton,
bullet paths, or plain prose.

Every response goes through six checks, and **a sample passes only if every check
produces zero errors** — score is 1 or 0, no partial credit. The headline number is the
pass rate.

| # | Check | Typical failure |
|---|-------|-----------------|
| 1 | **Parse** | truncation, trailing text, JSON inside a `yaml` fence |
| 2 | **Code block** | fence required but absent |
| 3 | **No commentary** | text before/after the document |
| 4 | **Structure** | bare list vs `{"key": [...]}` wrapper, wrong key name |
| 5 | **Item count** | exact `n`, or a `[min, max]` range |
| 6 | **Schema** | types, required fields, enums, numeric bounds, **extraneous keys** |

Two details in Liquid's validator are worth internalising now, because they shape both
the training data and the rewards:

* **`json_schema` is always `{"type": "array", ...}`.** It describes the *unwrapped*
  list of items — never the wrapper object. The validator unwraps a single-key dict
  before the schema walk.
* **A YAML request cannot be satisfied with JSON.** YAML is a superset of JSON, so JSON
  parses fine as YAML; the validator explicitly rejects flow-style mappings and demands
  real block-style YAML.

In [ ]:
from ifstruct.validator import validate_response
import ifstruct, pathlib

# The frozen test set ships inside the installed package's repo; if pip installed it
# without the data dir, fall back to fetching the jsonl straight from GitHub.
def load_ifstruct_test():
    here = pathlib.Path(ifstruct.__file__).resolve().parent.parent / "data" / "test.jsonl"
    if here.exists():
        raw = here.read_text()
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/Liquid4All/ifstruct/main/data/test.jsonl"
        raw = urllib.request.urlopen(url).read().decode()
    return [json.loads(line) for line in raw.splitlines() if line.strip()]

TEST_ROWS = load_ifstruct_test()
print(f"IFStruct test rows: {len(TEST_ROWS)}")
print("columns:", sorted(TEST_ROWS[0].keys()))

for col in ["output_format", "require_wrapper_key", "require_code_block", "require_no_commentary"]:
    print(f"  {col:24s} {dict(Counter(r[col] for r in TEST_ROWS))}")
print("  schema root types      ",
      dict(Counter(r["json_schema"].get("type") for r in TEST_ROWS)))

In [ ]:
# Look at one real example end to end.
row = next(r for r in TEST_ROWS if r["output_format"] == "json" and len(r["prompt"]) < 1200)

print("REQUIREMENTS:", {k: v for k, v in row.items() if k not in ("prompt", "json_schema")})
print("\n--- PROMPT " + "-" * 70)
print(row["prompt"])
print("\n--- SCHEMA " + "-" * 70)
print(json.dumps(row["json_schema"], indent=2)[:900], "...")

### Build a provably-passing answer, then break it

The fastest way to understand a benchmark is to write a perfect answer to it and then
watch which mutation kills which check. `reference_answer()` below fills a schema with
dummy values and renders it in the requested shape — it is not a model, it is a
constructive proof that the row is satisfiable.

We will reuse it in section 6 to prove that every row of our *synthetic training set*
is solvable, which is the difference between a training set and a wish.

In [ ]:
import yaml

def _instance(schema, rng, depth=0):
    """Fabricate one value satisfying a (sub)schema."""
    if "enum" in schema:
        return rng.choice(schema["enum"])
    t = schema.get("type")
    if t == "array":
        lo = schema.get("minItems", 1)
        hi = schema.get("maxItems", lo)
        return [_instance(schema["items"], rng, depth + 1) for _ in range(rng.randint(lo, hi))]
    if t == "object":
        return {k: _instance(v, rng, depth + 1) for k, v in schema.get("properties", {}).items()}
    if t == "integer":
        return rng.randint(int(schema.get("minimum", 0)), int(schema.get("maximum", 100)))
    if t == "number":
        return round(rng.uniform(float(schema.get("minimum", 0)), float(schema.get("maximum", 100))), 2)
    if t == "boolean":
        return rng.choice([True, False])
    # deliberately include quotes/newlines: escaping is one of IFStruct's hard axes
    return rng.choice(["Sample text", 'He said "ok" and left', "line one\nline two"])


def reference_answer(row, rng=None):
    """Render a response that scores 1.0 under the official validator."""
    rng = rng or random.Random(row.get("seed", 0))
    schema = row["json_schema"]
    schema = json.loads(schema) if isinstance(schema, str) else schema
    count = row["top_level_count"]
    count = json.loads(count) if isinstance(count, str) else count

    items = _instance(schema, rng)
    n = count if isinstance(count, int) else rng.randint(count[0], count[1])
    while len(items) < n:
        items.append(_instance(schema["items"], rng))
    items = items[:n]

    payload = {row["top_level_key"]: items} if row["require_wrapper_key"] else items
    if row["output_format"] == "json":
        body, lang = json.dumps(payload, indent=2), "json"
    else:
        # block style only -- flow-style YAML is rejected as "JSON in disguise"
        body = yaml.safe_dump(payload, default_flow_style=False, sort_keys=False,
                              allow_unicode=True, width=10**6).rstrip()
        lang = "yaml"
    return f"```{lang}\n{body}\n```" if row["require_code_block"] else body


def score_one(row, response):
    """Run the official IFStruct validator on a single response."""
    schema = row["json_schema"]
    count = row["top_level_count"]
    return validate_response(
        response=response,
        json_schema=json.loads(schema) if isinstance(schema, str) else schema,
        top_level_count=json.loads(count) if isinstance(count, str) else count,
        require_no_commentary=bool(row["require_no_commentary"]),
        output_format=row["output_format"],
        top_level_key=row["top_level_key"],
        require_wrapper_key=bool(row["require_wrapper_key"]),
        require_code_block=bool(row["require_code_block"]),
    )


# Sanity check the harness against ALL 2,000 real rows: a constructed answer must
# always pass. If this is not 1.000, our understanding of the spec is wrong.
fails = sum(0 if score_one(r, reference_answer(r)).passed else 1 for r in TEST_ROWS)
print(f"reference answers passing on the real test set: {(len(TEST_ROWS)-fails)/len(TEST_ROWS):.3f}"
      f"  ({fails} failures)")

In [ ]:
# Now mutate a perfect answer and watch the checks fall over one at a time.
demo = next(r for r in TEST_ROWS
            if r["output_format"] == "json" and r["require_wrapper_key"]
            and r["require_code_block"] and r["require_no_commentary"])
perfect = reference_answer(demo, random.Random(0))
items = json.loads(perfect.split("```json\n")[1].rsplit("\n```", 1)[0])[demo["top_level_key"]]
fence = lambda obj: "```json\n" + json.dumps(obj, indent=2) + "\n```"

mutations = {
    "perfect":                    perfect,
    "chatty preamble":            "Sure! Here you go:\n\n" + perfect + "\n\nHope that helps!",
    "fence stripped":             perfect.replace("```json\n", "").replace("\n```", ""),
    "wrapper dropped":            fence(items),
    "wrong wrapper key":          fence({"data": items}),
    "item count doubled":         fence({demo["top_level_key"]: items * 2}),
    "extra key on each item":     fence({demo["top_level_key"]: [dict(i, note="fyi") for i in items]}),
    "empty list":                 fence({demo["top_level_key"]: []}),
    "truncated mid-object":       "```json\n" + json.dumps({demo["top_level_key"]: items})[:60],
    "schema echoed back":         fence(demo["json_schema"]),
}

print(f"{'mutation':26s} {'pass':>5s}   first error")
print("-" * 100)
for label, text in mutations.items():
    v = score_one(demo, text)
    err = v.errors[0][:62] if v.errors else ""
    print(f"{label:26s} {str(v.passed):>5s}   {err}")

## 4. Load LFM2.5-350M and attach a LoRA adapter

### Do not reach for 4-bit reflexively

350M parameters in fp16 is **~0.7 GB**. A T4 has ~15 GB. Quantising a model this small
buys you memory you were never short of, while costing generation speed (GRPO's
bottleneck is sampling `G` completions per prompt) and a little accuracy. `LOAD_IN_4BIT`
defaults to `False` for that reason; the 4-bit path is kept below for when you genuinely
need it.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant_config = None
if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=DTYPE,       # NOT hard-coded bf16: a T4 cannot do bf16
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=DTYPE,             # transformers v5 renamed `torch_dtype` -> `dtype`
    device_map="auto",
)
model.config.use_cache = False       # incompatible with gradient checkpointing

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params/1e6:.1f}M params, arch={model.config.model_type}")
print(f"weights on GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# The chat template. The official IFStruct harness sends a SINGLE user message and no
# system prompt -- we mirror that everywhere so our numbers stay comparable to theirs.
example_msgs = [{"role": "user", "content": "Give me two recipes as JSON."}]
print(tokenizer.apply_chat_template(example_msgs, tokenize=False, add_generation_prompt=True))

### Find the real LoRA target names — don't copy them from a blog post

LFM2 is a hybrid stack: some layers are attention, some are short convolutions, chosen
per-layer by `config.layer_types`. Its attention output projection is named **`out_proj`,
not `o_proj`** — the name most LoRA snippets on the internet use. PEFT silently ignores
target names that match nothing, so a copied list can leave whole blocks untrained and
your loss looking mysteriously flat.

Enumerate the modules instead of trusting a list:

In [ ]:
import torch.nn as nn
from collections import defaultdict

leaf_linears = defaultdict(int)
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) or module.__class__.__name__ in ("Linear4bit", "Linear8bitLt"):
        leaf_linears[name.split(".")[-1]] += 1

print("linear submodule names in this checkpoint:")
for n, c in sorted(leaf_linears.items(), key=lambda kv: -kv[1]):
    print(f"  {n:12s} x{c}")

# Everything except the LM head. For LFM2 this resolves to:
#   attention : q_proj, k_proj, v_proj, out_proj
#   short conv: in_proj, out_proj
#   MLP       : w1, w2, w3
TARGET_MODULES = sorted(n for n in leaf_linears if n != "lm_head")
print("\nLoRA targets ->", TARGET_MODULES)
print("note: 'o_proj' present?", "o_proj" in leaf_linears, "  'out_proj' present?", "out_proj" in leaf_linears)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if LOAD_IN_4BIT:
    # casts norms/embeddings to fp32 and enables input grads -- required before LoRA
    # on a quantised base, otherwise gradient checkpointing detaches the graph.
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, peft_config)
model.enable_input_require_grads()   # needed for gradient checkpointing + frozen base
model.print_trainable_parameters()

Aim for roughly **1–3% trainable**. If the run refuses to move at all, widen the targets
or raise `r` before you touch the learning rate.

> **A useful property of LoRA:** `lora_B` is initialised to zeros, so a freshly attached
> adapter is a mathematical no-op. The model right now *is* the base model. That is why
> we can measure our baseline through the wrapped model — and why, after training, we can
> A/B the two with `model.disable_adapter()` instead of reloading a second copy.

## 5. Measure the baseline

Two rules make a before/after comparison trustworthy:

1. **Score the identical rows both times** (`EVAL_SEED` is fixed).
2. **Generate under identical settings both times** — same `max_new_tokens`, greedy
   decoding, single user message, no system prompt. That last point matters: adding a
   helpful *"return only valid JSON"* system prompt would lift the number for free and
   make it incomparable to the published baseline, which sends the bare user turn.

Structured output is decoded **cold**: `do_sample=False`. Sampling is for training-time
exploration only.

In [ ]:
@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE):
    """Greedy, left-padded batched generation. One user message, matching the harness."""
    model.eval()
    prev_side, prev_trunc = tokenizer.padding_side, tokenizer.truncation_side
    tokenizer.padding_side = "left"          # right padding corrupts generation
    tokenizer.truncation_side = "left"       # never truncate away the generation prompt
    outputs = []
    try:
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                                   tokenize=False, add_generation_prompt=True)
                     for p in chunk]
            # add_special_tokens=False: the chat template already emitted BOS
            enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                            max_length=2048, add_special_tokens=False).to(model.device)
            gen = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True,
            )
            new_tokens = gen[:, enc["input_ids"].shape[-1]:]
            outputs.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))
            print(f"  generated {min(i+batch_size, len(prompts))}/{len(prompts)}", end="\r")
    finally:
        tokenizer.padding_side, tokenizer.truncation_side = prev_side, prev_trunc
    print()
    return outputs


def evaluate(rows, label, **gen_kwargs):
    """Generate + score with the official validator. Returns (pass_rate, per-row records)."""
    t0 = time.time()
    completions = generate_batch([r["prompt"] for r in rows], **gen_kwargs)
    records = []
    for r, c in zip(rows, completions):
        v = score_one(r, c)
        records.append({"row": r, "completion": c, "passed": v.passed, "errors": v.errors,
                        "ratio": v.details.get("schema_match_ratio", 0.0)})
    rate = sum(x["passed"] for x in records) / len(records)
    print(f"\n=== {label}: {sum(x['passed'] for x in records)}/{len(records)} passed "
          f"({100*rate:.1f}%)   [{time.time()-t0:.0f}s]")
    return rate, records


def report(records, label):
    """Slice the pass rate the way the official harness does."""
    print(f"\n--- {label} ---")
    for axis, fn in [("format",    lambda r: r["row"]["output_format"]),
                     ("structure", lambda r: "wrapper" if r["row"]["require_wrapper_key"] else "bare list"),
                     ("fenced",    lambda r: f"fence={r['row']['require_code_block']}")]:
        groups = {}
        for rec in records:
            groups.setdefault(fn(rec), []).append(rec["passed"])
        line = "  ".join(f"{k}: {100*sum(v)/len(v):5.1f}% (n={len(v)})" for k, v in sorted(groups.items()))
        print(f"  by {axis:10s} {line}")
    errs = Counter()
    for rec in records:
        for e in rec["errors"]:
            errs[re.sub(r"['\"].*", "", e)[:58]] += 1
    print("  most common errors:")
    for e, c in errs.most_common(6):
        print(f"    {c:4d}  {e}")

In [ ]:
rng_eval = random.Random(EVAL_SEED)
EVAL_ROWS = rng_eval.sample(TEST_ROWS, N_EVAL)

baseline_rate, baseline_records = evaluate(EVAL_ROWS, "BASELINE (LFM2.5-350M, adapter is a no-op)")
report(baseline_records, "baseline breakdown")

In [ ]:
# Read a couple of failures before you train. This is the highest-value five minutes
# in the whole notebook: it tells you which reward is going to do the work.
shown = 0
for rec in baseline_records:
    if not rec["passed"] and shown < 2:
        shown += 1
        print("=" * 100)
        print("REQUIRED:", {k: rec["row"][k] for k in
              ("output_format", "require_wrapper_key", "require_code_block",
               "require_no_commentary", "top_level_count", "top_level_key")})
        print("ERRORS  :", rec["errors"][:3])
        print("-" * 100)
        print(rec["completion"][:700])

## 6. Build a training set

**The public IFStruct split is test-only and frozen.** Training on any slice of it would
contaminate the number you are trying to move. So we generate our own, mirroring how
Liquid describes building IFStruct: entity templates × a schema builder × several
presentation styles × the constraint axes.

The generator below produces rows with the *exact* column contract the validator expects,
so training rewards and eval scoring run through identical code.

What we deliberately vary, because that is what makes the skill transfer rather than
memorise:

* **format** — JSON / block YAML (50/50)
* **container** — bare list vs `{"key": [...]}` wrapper
* **fencing** — required / forbidden-in-effect (~63% fenced, matching the test set)
* **commentary** — allowed / banned
* **count** — exact (`2`) and ranges (`[1, 3]`)
* **presentation** — markdown table, bullets, raw JSON Schema, pseudo-block, prose
* **hard types** — nested object arrays, enums, numeric bounds, booleans, and strings
  that must carry embedded quotes and newlines (the escaping axis)

In [ ]:
import json, random
from typing import Any

# ----------------------------------------------------------------------------- entities
def F(name, type_, **kw):
    d = {"name": name, "type": type_}
    d.update(kw)
    return d

ENTITY_SPECS = [
    {
        "key": "invoice", "label": "invoices", "singular": "invoice",
        "scenarios": ["a freelance design studio billing a retail client",
                      "a cloud hosting provider billing a startup",
                      "a logistics company billing a manufacturer"],
        "context": "Amounts should add up plausibly and identifiers should look like real invoice references.",
        "fields": [
            F("invoice_number", "string", desc="Invoice reference as printed on the document"),
            F("currency", "enum", values=["USD", "EUR", "GBP", "JPY"]),
            F("subtotal", "number", min=10, max=250000),
            F("tax_rate_pct", "number", min=0, max=27),
            F("total_due", "number", min=10, max=300000),
            F("days_until_due", "integer", min=0, max=180),
            F("paid", "boolean"),
            F("line_items", "object_array", item_min=2, item_max=4, desc="Individual billed lines", items=[
                F("description", "string", desc="What was billed"),
                F("quantity", "integer", min=1, max=500),
                F("unit_price", "number", min=0, max=50000),
            ]),
        ],
    },
    {
        "key": "gpu_review", "label": "GPU reviews", "singular": "GPU review",
        "scenarios": ["a mid-range card tested for 1440p gaming",
                      "a workstation card tested for CUDA rendering",
                      "a last-generation card tested for local LLM inference"],
        "context": "Benchmark numbers should be plausible for the class of card being described.",
        "fields": [
            F("model_name", "string", desc="Card name only, no vendor marketing suffix"),
            F("vram_gb", "integer", min=4, max=96),
            F("avg_fps_1440p", "number", min=10, max=400),
            F("power_draw_w", "integer", min=50, max=700),
            F("verdict", "enum", values=["buy", "wait", "skip"]),
            F("supports_ray_tracing", "boolean"),
            F("pros", "string_array", item_min=2, item_max=3, desc="Short positive points"),
        ],
    },
    {
        "key": "clinical_trial", "label": "clinical trial records", "singular": "clinical trial record",
        "scenarios": ["a phase II oncology study", "a phase III cardiology study",
                      "an early-phase vaccine immunogenicity study"],
        "context": "Enrollment figures and phase labels should be internally consistent.",
        "fields": [
            F("trial_title", "string", desc="Official study title"),
            F("phase", "enum", values=["I", "I/II", "II", "II/III", "III", "IV"]),
            F("enrolled_participants", "integer", min=8, max=15000),
            F("randomized", "boolean"),
            F("primary_endpoint", "string", desc="Primary outcome measure"),
            F("arms", "object_array", item_min=2, item_max=3, desc="Study arms", items=[
                F("arm_label", "string"),
                F("allocation", "enum", values=["treatment", "placebo", "active_comparator", "observation"]),
                F("participants", "integer", min=4, max=8000),
            ]),
        ],
    },
    {
        "key": "job_posting", "label": "job postings", "singular": "job posting",
        "scenarios": ["a remote backend role at a fintech scale-up",
                      "an on-site hardware role at a robotics company",
                      "a hybrid data role at a healthcare provider"],
        "context": "Compensation bands and seniority should be consistent with each other.",
        "fields": [
            F("job_title", "string", desc="Role title only, no company name"),
            F("seniority", "enum", values=["intern", "junior", "mid", "senior", "staff", "principal"]),
            F("salary_min_usd", "integer", min=20000, max=400000),
            F("salary_max_usd", "integer", min=25000, max=600000),
            F("remote_allowed", "boolean"),
            F("required_skills", "string_array", item_min=3, item_max=5, desc="Named technologies or skills"),
        ],
    },
    {
        "key": "conference_schedule", "label": "conference schedules", "singular": "conference schedule",
        "scenarios": ["a two-track systems conference", "a single-track design summit",
                      "an academic workshop day"],
        "context": "Session times and room labels should be plausible for a real event programme.",
        "fields": [
            F("track_name", "string", desc="Track or room programme name"),
            F("day_index", "integer", min=1, max=5),
            F("livestreamed", "boolean"),
            F("sessions", "object_array", item_min=2, item_max=4, desc="Talks in this track", items=[
                F("title", "string"),
                F("speaker", "string"),
                F("minutes", "integer", min=5, max=180),
                F("session_type", "enum", values=["keynote", "talk", "panel", "workshop", "lightning"]),
            ]),
        ],
    },
    {
        "key": "recipe", "label": "recipes", "singular": "recipe",
        "scenarios": ["a weeknight one-pan dinner", "a bakery-style breakfast pastry",
                      "a vegetarian batch-cook lunch"],
        "context": "Quantities and timings should be realistic for a home kitchen.",
        "fields": [
            F("title", "string", desc="Dish name only"),
            F("servings", "integer", min=1, max=12),
            F("total_minutes", "integer", min=5, max=480),
            F("difficulty", "enum", values=["easy", "medium", "hard"]),
            F("vegetarian", "boolean"),
            F("ingredients", "string_array", item_min=3, item_max=6, desc="Ingredient lines with quantities"),
            F("steps", "string_array", item_min=3, item_max=5, desc="Ordered preparation steps"),
        ],
    },
    # --- escaping-heavy entities (strings that must carry quotes / newlines / symbols)
    {
        "key": "bug_report_batch", "label": "bug reports", "singular": "bug report",
        "scenarios": ["a crash triage queue for a desktop app",
                      "a regression sweep after a dependency bump",
                      "an on-call queue for a payments service"],
        "context": ("Log excerpts should look like real captured output. Include quotation marks and literal "
                    "newline characters inside the excerpt text where natural."),
        "escaping": True,
        "fields": [
            F("summary", "string", desc="One-line issue summary"),
            F("severity", "enum", values=["blocker", "critical", "major", "minor", "trivial"]),
            F("reproducible", "boolean"),
            F("occurrences_last_7d", "integer", min=1, max=50000),
            F("log_excerpt", "string", desc='Raw captured log lines, including quotes and newlines'),
        ],
    },
    {
        "key": "dialogue_sample", "label": "dialogue samples", "singular": "dialogue sample",
        "scenarios": ["a support call transcript", "a two-hander scene from a stage play",
                      "an interview excerpt"],
        "context": ("Utterances should read naturally and contain apostrophes and quoted speech where the "
                    "exchange calls for it."),
        "escaping": True,
        "fields": [
            F("scene_label", "string", desc="Short label for the exchange"),
            F("register", "enum", values=["formal", "casual", "technical", "confrontational"]),
            F("turns", "object_array", item_min=3, item_max=5, desc="Speaking turns in order", items=[
                F("speaker", "string"),
                F("utterance", "string", desc='What the speaker says, including any quoted speech'),
                F("interrupted", "boolean"),
            ]),
        ],
    },
    {
        "key": "config_snippet_audit", "label": "config audits", "singular": "config audit",
        "scenarios": ["a hardening review of a reverse proxy", "a review of a CI pipeline definition",
                      "a review of a container runtime configuration"],
        "context": ("Config excerpts should be copied-out fragments with indentation, symbols and quoting "
                    "preserved exactly."),
        "escaping": True,
        "fields": [
            F("file_path", "string", desc="Path of the audited file"),
            F("risk_level", "enum", values=["none", "low", "medium", "high", "critical"]),
            F("auto_fixable", "boolean"),
            F("findings", "object_array", item_min=2, item_max=3, desc="Individual audit findings", items=[
                F("rule_id", "string"),
                F("excerpt", "string", desc="The offending configuration lines, verbatim"),
                F("line_number", "integer", min=1, max=20000),
            ]),
        ],
    },
    {
        "key": "api_endpoint_spec", "label": "API endpoint specs", "singular": "API endpoint spec",
        "scenarios": ["an internal billing service", "a public read-only catalogue API",
                      "an admin service behind mTLS"],
        "context": "Paths, methods and status codes should be coherent with each other.",
        "fields": [
            F("path", "string", desc="URL path template"),
            F("method", "enum", values=["GET", "POST", "PUT", "PATCH", "DELETE"]),
            F("auth_required", "boolean"),
            F("timeout_ms", "integer", min=50, max=120000),
            F("parameters", "object_array", item_min=2, item_max=4, desc="Accepted parameters", items=[
                F("name", "string"),
                F("location", "enum", values=["path", "query", "header", "body"]),
                F("required", "boolean"),
            ]),
        ],
    },
]

In [ ]:
# ----------------------------------------------------------------------------- schema
def _leaf_schema(f: dict) -> dict[str, Any]:
    t = f["type"]
    if t == "enum":
        s = {"type": "string", "enum": list(f["values"])}
    elif t in ("integer", "number"):
        s = {"type": t}
        if "min" in f: s["minimum"] = f["min"]
        if "max" in f: s["maximum"] = f["max"]
    elif t == "boolean":
        s = {"type": "boolean"}
    elif t == "string_array":
        s = {"type": "array", "items": {"type": "string"},
             "minItems": f["item_min"], "maxItems": f["item_max"]}
    elif t == "object_array":
        s = {"type": "array",
             "items": {"type": "object",
                       "properties": {c["name"]: _leaf_schema(c) for c in f["items"]},
                       "required": [c["name"] for c in f["items"]]},
             "minItems": f["item_min"], "maxItems": f["item_max"]}
    else:
        s = {"type": "string"}
    if f.get("desc") and t not in ("object_array",):
        s.setdefault("description", f["desc"])
    return s


def build_schema(fields: list[dict], count) -> dict[str, Any]:
    """IFStruct schemas are ALWAYS `{"type": "array", "items": {...}}` — the schema
    describes the *unwrapped* list of items, never the wrapper object."""
    lo, hi = (count, count) if isinstance(count, int) else (count[0], count[1])
    return {
        "type": "array",
        "items": {"type": "object",
                  "properties": {f["name"]: _leaf_schema(f) for f in fields},
                  "required": [f["name"] for f in fields]},
        "minItems": lo, "maxItems": hi,
    }

In [ ]:
# ----------------------------------------------------------------------------- prompt pieces
def _constraint_text(f: dict) -> str:
    t = f["type"]
    if t == "enum":
        return "one of: " + ", ".join(f["values"])
    if t in ("integer", "number"):
        if "min" in f and "max" in f: return f"between {f['min']} and {f['max']}"
        return ""
    if t in ("string_array", "object_array"):
        return f"{f['item_min']}-{f['item_max']} items"
    return ""


def _type_word(f: dict) -> str:
    return {"enum": "string", "string_array": "array of strings",
            "object_array": "array of objects"}.get(f["type"], f["type"])


def _count_phrase(count) -> str:
    return str(count) if isinstance(count, int) else f"{count[0]}-{count[1]}"


def _pseudo_value(f: dict, indent: str) -> str:
    t = f["type"]
    if t == "enum":
        return "|".join(f["values"])
    if t in ("integer", "number"):
        bits = []
        if "min" in f: bits.append(f"≥{f['min']}")
        if "max" in f: bits.append(f"≤{f['max']}")
        return f"<{t}>" + (f" ({', '.join(bits)})" if bits else "")
    if t == "boolean":
        return "<boolean>"
    if t == "string_array":
        return f"\n{indent}  - <string>  # {f['item_min']}-{f['item_max']} items\n{indent}  - ..."
    if t == "object_array":
        inner = []
        for i, c in enumerate(f["items"]):
            lead = f"{indent}  - " if i == 0 else f"{indent}    "
            inner.append(f"{lead}{c['name']}: {_pseudo_value(c, indent + '    ')}")
        return (f"  # {f['item_min']}-{f['item_max']} items\n" + "\n".join(inner) + f"\n{indent}  - ...")
    return "<string>" + (f"  # {f['desc']}" if f.get("desc") else "")

In [ ]:
# ----------------------------------------------------------------------------- 5 presentation styles
def style_md_table(fields, count, key, wrapper):
    rows = ["| path | constraints | notes | type |", "| --- | --- | --- | --- |"]
    for f in fields:
        rows.append(f"| `{f['name']}` | {_constraint_text(f)} | {f.get('desc','')} | {_type_word(f)} |")
        for c in f.get("items", []):
            rows.append(f"| `{f['name']}[].{c['name']}` | {_constraint_text(c)} | {c.get('desc','')} | {_type_word(c)} |")
    return "Requested fields:\n" + "\n".join(rows)


def style_bullets(fields, count, key, wrapper):
    out = ["Fields to include:"]
    for f in fields:
        con = _constraint_text(f)
        out.append(f"- `{f['name']}` ({_type_word(f)})" + (f" — {con}" if con else "")
                   + (f". {f['desc']}." if f.get("desc") else ""))
        for c in f.get("items", []):
            con2 = _constraint_text(c)
            out.append(f"  - `{f['name']}[].{c['name']}` ({_type_word(c)})" + (f" — {con2}" if con2 else ""))
    return "\n".join(out)


def style_raw_jsonschema(fields, count, key, wrapper):
    return ("Conform to this JSON Schema:\n\n```\n"
            + json.dumps(build_schema(fields, count), indent=2) + "\n```")


def style_pseudo_block(fields, count, key, wrapper):
    lines = ["Response structure:", "```"]
    if wrapper:
        lines.append(f"{key}:  # {_count_phrase(count)} items")
        ind, lead = "  ", "  - "
    else:
        lines.append(f"# {_count_phrase(count)} items")
        ind, lead = "", "- "
    for i, f in enumerate(fields):
        prefix = (lead if i == 0 else ind + "  ")
        lines.append(f"{prefix}{f['name']}: {_pseudo_value(f, ind + '  ')}")
    lines.append(f"{ind}- ...")
    lines.append("```")
    return "\n".join(lines)


def style_prose(fields, count, key, wrapper):
    parts = []
    for f in fields:
        con = _constraint_text(f)
        seg = f"{f['name']} as {_type_word(f)}"
        if con: seg += f" ({con})"
        if f.get("items"):
            seg += " where each entry has " + ", ".join(
                c["name"] + (f" [{_constraint_text(c)}]" if _constraint_text(c) else "") for c in f["items"])
        parts.append(seg)
    return "Each entry needs " + "; ".join(parts) + "."


STYLES = [style_md_table, style_bullets, style_raw_jsonschema, style_pseudo_block, style_prose]

In [ ]:
# ----------------------------------------------------------------------------- row assembly
FORMAT_LINES = {
    ("json", True):  ["Output valid JSON. Return an object with `{key}` as the key for the array.",
                      "Format your response as JSON. Use `{key}` as the top-level key wrapping the array.",
                      "Your response should be JSON, wrapped in an object under the key `{key}`."],
    ("json", False): ["Output valid JSON. Return a bare list at the top level, not wrapped in an object.",
                      "Use JSON format for your response. Return a bare list at the top level, not wrapped in an object.",
                      "Provide the output in JSON as a top-level array (no wrapper object)."],
    ("yaml", True):  ["Format your response as YAML. Return an object with `{key}` as the key for the array.",
                      "Respond in YAML with `{key}` as the top-level key for the array.",
                      "Use YAML. The array should sit under the top-level key `{key}`."],
    ("yaml", False): ["Format your response as YAML. Return a bare list at the top level, not wrapped in an object.",
                      "Respond in block-style YAML as a bare top-level list, not wrapped in an object.",
                      "Use YAML for your response. Top level must be a plain list, no wrapper key."],
}
FENCE_LINES = {
    True: {"json": ["Put the JSON in a ```json fenced code block.", "Enclose the JSON in a ```json code block.",
                    "Wrap your response in a code block.", "Use a code block for your response."],
           "yaml": ["Put the YAML in a ```yaml fenced code block.", "Enclose the YAML in a ```yaml code block.",
                    "Format your response inside a ```yaml code block.", "Wrap your YAML in a ```yaml code block."]},
    False: {"json": ["Output plain JSON without wrapping in a code block.",
                     "No code block needed - output the JSON directly.",
                     "Output the raw JSON directly without code block fencing."],
            "yaml": ["Output plain YAML without wrapping in a code block.",
                     "No code block needed - output the YAML directly.",
                     "Output the raw YAML directly without code block fencing."]},
}
NO_COMMENT_LINES = {
    "json": ["Respond with just the JSON, no explanations.", "Just the JSON, no commentary or preamble.",
             "Return only the JSON document - no preface, no trailing notes."],
    "yaml": ["Respond with just the YAML, no explanations.", "Just the YAML, no commentary or preamble.",
             "Return only the YAML document - no preface, no trailing notes."],
}


def make_example(rng: random.Random, seed: int) -> dict[str, Any]:
    spec = rng.choice(ENTITY_SPECS)
    fmt = rng.choice(["json", "yaml"])
    wrapper = rng.random() < 0.5
    fence = rng.random() < 0.63          # ifstruct test set is ~63% fenced
    no_comment = rng.random() < 0.5
    count = rng.choice([1, 2, 3, 4, [1, 2], [1, 3], [2, 3], [2, 4], [3, 4]])

    # Keep a subset of fields so prompts vary in width (always >= 4).
    fields = list(spec["fields"])
    if len(fields) > 4 and rng.random() < 0.45:
        keep = rng.randint(4, len(fields))
        fields = fields[:keep]

    key = spec["key"] if rng.random() < 0.5 else spec["key"] + "s"
    schema = build_schema(fields, count)
    scenario = rng.choice(spec["scenarios"])

    head = (f"Generate {_count_phrase(count)} {spec['label'] if not isinstance(count, int) or count > 1 else spec['singular']} "
            f"for {scenario}.")
    body = [head, f"Assign field values consistent with {scenario}.", spec["context"]]

    fmt_line = rng.choice(FORMAT_LINES[(fmt, wrapper)]).format(key=key)
    lines = ["\n".join(body), "", fmt_line, rng.choice(FENCE_LINES[fence][fmt]), "",
             rng.choice(STYLES)(fields, count, key, wrapper)]
    if no_comment:
        lines += ["", rng.choice(NO_COMMENT_LINES[fmt])]
    if rng.random() < 0.35:
        lines += ["", "Match the requested schema exactly and do not add extra keys."]

    return {
        "seed": seed,
        "entity_type": f"train__{spec['key']}",
        "prompt": "\n".join(lines).strip(),
        "json_schema": schema,
        "top_level_count": count,
        "top_level_key": key,
        "require_wrapper_key": wrapper,
        "require_code_block": fence,
        "require_no_commentary": no_comment,
        "output_format": fmt,
    }


def build_trainset(n: int, seed: int = 0) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    return [make_example(rng, seed=i) for i in range(n)]

### Generate, then *prove the set is solvable*

A synthetic training set is worthless if its rows contradict themselves — a reward that
can never reach 1.0 teaches nothing, and GRPO with an all-zero group has no gradient at
all (every completion gets the same advantage of 0). So we run the constructive
`reference_answer()` over every row and require a 100% pass rate under the official
validator before we spend a single GPU-minute.

In [ ]:
train_rows = build_trainset(N_TRAIN, seed=0)
dev_rows   = build_trainset(N_DEV,   seed=99)

fails = [r for r in train_rows + dev_rows if not score_one(r, reference_answer(r)).passed]
print(f"solvability check: {len(train_rows)+len(dev_rows)-len(fails)}"
      f"/{len(train_rows)+len(dev_rows)} rows satisfiable")
assert not fails, f"{len(fails)} unsatisfiable rows -- fix the generator before training"

for col in ["output_format", "require_wrapper_key", "require_code_block", "require_no_commentary"]:
    print(f"  {col:24s} {dict(Counter(r[col] for r in train_rows))}")
print("  entity types            ", len(set(r["entity_type"] for r in train_rows)))

# Contamination guard: no generated prompt may coincide with a test prompt.
overlap = {r["prompt"] for r in train_rows} & {r["prompt"] for r in TEST_ROWS}
print("  overlap with test set   ", len(overlap))
assert not overlap

print("\n" + "=" * 100)
print(train_rows[3]["prompt"])

### Two Arrow gotchas when this becomes a `Dataset`

`datasets` is backed by Arrow, which needs one fixed type per column:

* **`json_schema`** — every row has a *different* nested shape. Arrow will try to unify
  them into one struct and either explode or silently null out fields.
* **`top_level_count`** — is an `int` on some rows and a `[min, max]` list on others.

Store both as **JSON strings** and parse them inside the reward. Skipping this is the
single most common way this pipeline breaks with a baffling `ArrowInvalid`.

We also hand TRL a **conversational** prompt (a message list). It applies the chat
template itself — the same one-user-message shape the evaluation uses.

In [ ]:
from datasets import Dataset

def to_hf_dataset(rows):
    return Dataset.from_list([{
        "prompt": [{"role": "user", "content": r["prompt"]}],   # conversational
        "json_schema":           json.dumps(r["json_schema"]),  # <- JSON string
        "top_level_count":       json.dumps(r["top_level_count"]),  # <- JSON string
        "output_format":         r["output_format"],
        "top_level_key":         r["top_level_key"],
        "require_wrapper_key":   bool(r["require_wrapper_key"]),
        "require_code_block":    bool(r["require_code_block"]),
        "require_no_commentary": bool(r["require_no_commentary"]),
    } for r in rows])

train_ds = to_hf_dataset(train_rows).shuffle(seed=42)
print(train_ds)

prompt_tokens = [len(tokenizer.apply_chat_template(x["prompt"], tokenize=True,
                                                   add_generation_prompt=True))
                 for x in train_ds.select(range(100))]
prompt_tokens.sort()
print(f"prompt length (tokens): p50={prompt_tokens[50]}  p95={prompt_tokens[94]}  max={prompt_tokens[-1]}")
print("-> keep max_prompt_length above p95 or prompts get left-truncated and the "
      "constraints at the top of the prompt silently disappear")

## 7. The reward functions — the actual teacher

This is where the learning signal comes from, so it is worth being precise about the
design.

**Principle 1 — reward and metric are the same code.** Every reward below is derived
from one call to Liquid's `validate_response()`. There is no second, approximate
re-implementation to drift out of sync with the benchmark.

**Principle 2 — one binary reward is not enough.** IFStruct scores pass/fail. If that
were our only signal, then early in training *every* completion in a group would score 0,
the group's advantages would all be zero, and **GRPO would receive no gradient at all**.
So we decompose the single pass/fail into a ladder of partial credit that a struggling
model can climb:

| reward | weight | what it teaches |
|---|---|---|
| `reward_parse` | 0.2 | close your braces; emit the format that was requested |
| `reward_fence` | 0.1 | fence when asked |
| `reward_no_commentary` | 0.1 | stop apologising and explaining |
| `reward_structure` | 0.2 | bare list vs wrapper key, and the right key name |
| `reward_count` | 0.2 | emit exactly *n* items |
| `reward_schema_fields` | **0.5** | *continuous*: fraction of leaf fields that validate |
| `reward_ifstruct_pass` | **1.0** | the benchmark metric itself |

`reward_schema_fields` is the curriculum's engine: it is the only *continuous* term, so
it separates "9 of 10 fields right" from "2 of 10" long before any completion passes
outright.

**Principle 3 — pay the validator once.** Seven reward callables × `G` completions ×
100 steps is a lot of parsing. TRL calls each function separately, so the checks are
memoised on `(completion, row spec)` and all seven read one cached result.

In [ ]:
import json, re
from functools import lru_cache
from ifstruct.validator import validate_response

_COUNT_ERR = re.compile(r"^Expected (\d+|\d+-\d+) items")
_STRUCT_ERR = ("Expected wrapped object", "Expected bare list", "Expected top-level key")


def completion_text(c) -> str:
    """GRPO hands back a string (standard dataset) or a message list (conversational)."""
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c and isinstance(c[-1], dict):
        return c[-1].get("content", "")
    return str(c)


@lru_cache(maxsize=8192)
def _validate(text, schema_json, count_json, fmt, key, wrap, fence, nocom):
    return validate_response(
        response=text,
        json_schema=json.loads(schema_json),
        top_level_count=json.loads(count_json),
        require_no_commentary=nocom,
        output_format=fmt,
        top_level_key=key,
        require_wrapper_key=wrap,
        require_code_block=fence,
    )


def judge(completions, json_schema, top_level_count, output_format,
          top_level_key, require_wrapper_key, require_code_block, require_no_commentary):
    """One official validation per (completion, row); cached so the 7 rewards cost 1 pass."""
    out = []
    for c, sc, ct, fmt, key, wrap, fence, nocom in zip(
            completions, json_schema, top_level_count, output_format,
            top_level_key, require_wrapper_key, require_code_block, require_no_commentary):
        out.append(_validate(completion_text(c), sc, ct, fmt, key, bool(wrap), bool(fence), bool(nocom)))
    return out


def _parsed(v) -> bool:
    return bool(v.details.get("json_valid", v.details.get("yaml_valid", False)))


def _mk(fn):
    def reward(completions, json_schema, top_level_count, output_format, top_level_key,
               require_wrapper_key, require_code_block, require_no_commentary, **kwargs):
        results = judge(completions, json_schema, top_level_count, output_format,
                        top_level_key, require_wrapper_key, require_code_block, require_no_commentary)
        return [fn(v, fence) for v, fence in zip(results, require_code_block)]
    return reward


# --- stage 1: does it parse at all, in the format that was asked for?
reward_parse = _mk(lambda v, fence: float(v.details.get("json_valid", v.details.get("yaml_valid", False))))
# --- stage 2: fenced when a fence was demanded
reward_fence = _mk(lambda v, fence: 1.0 if (not fence or v.details.get("uses_code_block", False)) else 0.0)
# --- stage 3: nothing outside the document
reward_no_commentary = _mk(lambda v, fence: float(v.details.get("no_commentary", True)) if _parsed(v) else 0.0)
# --- stage 4: bare list vs wrapper key   (gated on a successful parse: an unparseable
#     completion never reaches the structure check, so it must not collect free credit)
reward_structure = _mk(lambda v, fence: 1.0 if _parsed(v) and not any(e.startswith(_STRUCT_ERR) for e in v.errors) else 0.0)
# --- stage 5: right number of items      (gated for the same reason)
reward_count = _mk(lambda v, fence: 1.0 if _parsed(v) and not any(_COUNT_ERR.match(e) for e in v.errors) else 0.0)
# --- stage 6: dense partial credit over every leaf field (the curriculum driver)
reward_schema_fields = _mk(lambda v, fence: float(v.details.get("schema_match_ratio", 0.0)))
# --- stage 7: the benchmark metric itself
reward_ifstruct_pass = _mk(lambda v, fence: float(v.passed))

for _f, _n in [(reward_parse, "reward_parse"), (reward_fence, "reward_fence"),
               (reward_no_commentary, "reward_no_commentary"), (reward_structure, "reward_structure"),
               (reward_count, "reward_count"), (reward_schema_fields, "reward_schema_fields"),
               (reward_ifstruct_pass, "reward_ifstruct_pass")]:
    _f.__name__ = _n

REWARD_FUNCS = [reward_parse, reward_fence, reward_no_commentary, reward_structure,
                reward_count, reward_schema_fields, reward_ifstruct_pass]
REWARD_WEIGHTS = [0.2, 0.1, 0.1, 0.2, 0.2, 0.5, 1.0]

### Test the rewards before you train on them

A reward function you have not adversarially tested is a bug you will pay for in GPU
hours. Score a perfect answer and a series of realistic failures, and read the ladder
top to bottom.

In [ ]:
def reward_row_columns(row, n=1):
    """Pack one row into the column-lists TRL will pass to the reward functions."""
    return dict(
        json_schema=[json.dumps(row["json_schema"])] * n,
        top_level_count=[json.dumps(row["top_level_count"])] * n,
        output_format=[row["output_format"]] * n,
        top_level_key=[row["top_level_key"]] * n,
        require_wrapper_key=[bool(row["require_wrapper_key"])] * n,
        require_code_block=[bool(row["require_code_block"])] * n,
        require_no_commentary=[bool(row["require_no_commentary"])] * n,
    )

names = [f.__name__.replace("reward_", "") for f in REWARD_FUNCS]
print(f"{'completion':28s} " + " ".join(f"{n[:8]:>8s}" for n in names) + "   weighted")
print("-" * 105)
for label, text in mutations.items():
    s = [f([text], **reward_row_columns(demo))[0] for f in REWARD_FUNCS]
    w = sum(v * wt for v, wt in zip(s, REWARD_WEIGHTS))
    print(f"{label:28s} " + " ".join(f"{v:8.2f}" for v in s) + f"   {w:7.3f}")

print("\nRead this table as the curriculum: a model that only learns to close its braces")
print("still climbs off the floor, and the two classic reward hacks -- an empty list and")
print("echoing the schema back -- sit near the bottom, not the top.")

> **On reward hacking.** GRPO optimises what you measure. The two degenerate strategies
> for this task are *emit an empty-but-valid container* and *echo the schema back as
> data*. Both are already punished here, because Liquid's validator enforces the item
> count and flags extraneous keys — the `empty list` and `schema echoed back` rows above
> score 0 on the terms that carry the most weight. If you swap in a laxer validator,
> re-run this cell and check those two rows first.
>
> Note what the table also shows: an extra key costs almost nothing on the *dense*
> term (one failed check among many, 1.00 → 0.97). It is `reward_ifstruct_pass`,
> at weight 1.0, that makes it expensive. Dense shaping rewards get you off the
> floor; only the binary term encodes the actual contract.

## 8. GRPO configuration for 100 steps on a T4

### How the batch numbers fit together

TRL requires that the **generation batch** divides evenly into groups:

```
generation_batch = per_device_bs × grad_accum × num_processes = 4 × 5 × 1 = 20 completions
prompts per step = 20 / num_generations = 20 / 4                          =  5 prompts
total prompts    = 5 × 100 steps                                          = 500 prompts
```

which is exactly the article's *"~500 samples, 100 steps"* budget — one clean epoch.
Memory is bounded by `per_device_bs` (4 sequences at a time), not by the 20.

### Choices worth understanding rather than copying

* **`beta=0.0`** — no KL term, so **no reference model is loaded**. Roughly halves
  memory. With only 100 steps there is little time to drift far from the base policy
  anyway.
* **`loss_type="dapo"`** — normalises over active tokens in the global batch instead of
  per-sequence, removing GRPO's well-known bias toward long completions.
* **`temperature=0.9`** — training *needs* diversity. If every completion in a group is
  identical, the advantages are all zero and the step is wasted. (Inference stays greedy.)
* **`learning_rate=2e-5`** — note this is **higher** than the 5e-6 typical of full-
  parameter GRPO. LoRA updates a ~1–3% slice of the weights through a low-rank
  bottleneck; at 5e-6 with a 100-step budget you will often see a flat reward curve and
  conclude the method does not work. If your curve is noisy, come *down*; if it is flat,
  go *up*.
* **`mask_truncated_completions=True`** — a completion cut off at the token cap is not
  evidence about anything; excluding it stops the model learning that trailing off is an
  acceptable ending.
* **`remove_unused_columns=False`** — **essential**. Without it, TRL drops
  `json_schema`, `top_level_count` and the flags, and your rewards get a `KeyError`.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    # --- the 100-step / 500-prompt budget -----------------------------------------
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    # --- optimisation --------------------------------------------------------------
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="constant_with_warmup",
    warmup_ratio=0.1,
    max_grad_norm=0.2,
    beta=0.0,                       # no reference model -> less VRAM
    loss_type="dapo",               # length-bias-free aggregation
    scale_rewards="group",          # advantage = (r - group mean) / group std
    reward_weights=REWARD_WEIGHTS,
    # --- sampling ------------------------------------------------------------------
    temperature=0.9,
    top_p=1.0,
    max_prompt_length=1536,
    max_completion_length=MAX_COMPLETION_LEN,
    mask_truncated_completions=True,
    # --- memory --------------------------------------------------------------------
    bf16=BF16_OK,
    fp16=not BF16_OK,               # T4 = fp16; forcing bf16 on Turing is a classic bug
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",
    # --- bookkeeping ---------------------------------------------------------------
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    log_completions=True,
    num_completions_to_print=2,
    remove_unused_columns=False,    # ESSENTIAL: rewards need the metadata columns
)

gen_batch = PER_DEVICE_BS * GRAD_ACCUM
assert gen_batch % NUM_GENERATIONS == 0, "generation batch must divide into whole groups"
print(f"{gen_batch} completions/step = {gen_batch // NUM_GENERATIONS} prompts x "
      f"{NUM_GENERATIONS} generations   ->   {MAX_STEPS * gen_batch // NUM_GENERATIONS} prompts total")

In [ ]:
trainer = GRPOTrainer(
    model=model,                    # already wrapped with LoRA
    args=grpo_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    peft_config=None,               # do NOT pass it twice
)

torch.cuda.empty_cache()
train_result = trainer.train()

### Reading the run

Watch three curves, in this order — they move in a predictable sequence:

1. **`rewards/reward_parse/mean` rises first.** Producing well-formed output is the
   easiest thing to learn and the prerequisite for everything else.
2. **`rewards/reward_schema_fields/mean` follows.** The dense term grinds upward as
   types, enums and bounds fall into line.
3. **`rewards/reward_ifstruct_pass/mean` moves last**, and in steps — it only counts
   completions that got *everything* right simultaneously.

`completions/mean_length` should **stabilise**, not grow. A steadily climbing length with
flat reward is the signature of a model padding its way toward a reward it cannot reach.

In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "reward" in h]
assert hist, "no reward logs -- did training run?"

def curve(key):
    """(steps, values) for one logged key, skipping entries where it is absent."""
    pts = [(h["step"], h[key]) for h in hist if h.get(key) is not None]
    return [p[0] for p in pts], [p[1] for p in pts]

reward_keys = sorted({k for h in hist for k in h
                      if k.startswith("rewards/") and k.endswith("/mean")})
len_key = next((k for h in hist for k in h if "completion" in k and "length" in k), None)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(*curve("reward"), lw=2, color="black")
axes[0].set_title("total weighted reward")

for k in reward_keys:
    axes[1].plot(*curve(k), lw=1.6, label=k.split("/")[1].replace("reward_", ""))
axes[1].set_title("reward components"); axes[1].legend(fontsize=7, ncol=2)

if len_key:
    axes[2].plot(*curve(len_key), color="tab:orange", lw=2)
    axes[2].set_title(len_key)
for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Measure again — same rows, same decoding

Nothing about the evaluation changes except the adapter. Because LoRA is additive we can
also A/B the *same* weights in memory: `model.disable_adapter()` gives back the base
policy exactly.

In [ ]:
model.config.use_cache = True     # re-enable KV cache for fast generation
tuned_rate, tuned_records = evaluate(EVAL_ROWS, "AFTER GRPO (adapter active)")
report(tuned_records, "post-GRPO breakdown")

In [ ]:
import math

def exact_mcnemar(before, after):
    """Paired two-sided sign test over the rows whose verdict changed."""
    b = sum(1 for x, y in zip(before, after) if x and not y)   # regressions
    c = sum(1 for x, y in zip(before, after) if y and not x)   # fixes
    n = b + c
    if n == 0:
        return b, c, 1.0
    k = min(b, c)
    tail = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return b, c, min(1.0, 2 * tail)

before = [r["passed"] for r in baseline_records]
after   = [r["passed"] for r in tuned_records]
regressed, fixed, p = exact_mcnemar(before, after)

print(f"{'':22s}{'baseline':>10s}{'tuned':>10s}{'delta':>10s}")
print("-" * 52)
print(f"{'IFStruct pass rate':22s}{100*baseline_rate:9.1f}%{100*tuned_rate:9.1f}%"
      f"{100*(tuned_rate-baseline_rate):+9.1f}")
mean_ratio = lambda recs: sum(r["ratio"] for r in recs) / len(recs)
print(f"{'mean field-match':22s}{100*mean_ratio(baseline_records):9.1f}%"
      f"{100*mean_ratio(tuned_records):9.1f}%"
      f"{100*(mean_ratio(tuned_records)-mean_ratio(baseline_records)):+9.1f}")
print(f"\nrows fixed: {fixed}   rows regressed: {regressed}   "
      f"exact paired sign test p = {p:.4f}")
print("\nPublished reference (full 2,000 rows, 8k token cap): 22.6% -> 29.7%")
print("Your absolute numbers will differ (subset + tighter token cap); the DELTA is the result.")

In [ ]:
# Where did the gain come from? Compare the error histograms.
def error_hist(records):
    c = Counter()
    for rec in records:
        for e in rec["errors"]:
            c[re.sub(r"['\"].*", "", e)[:52]] += 1
    return c

hb, ha = error_hist(baseline_records), error_hist(tuned_records)
keys = sorted(set(hb) | set(ha), key=lambda k: -(hb[k] + ha[k]))[:12]
print(f"{'error category':56s}{'before':>8s}{'after':>8s}{'delta':>8s}")
print("-" * 82)
for k in keys:
    print(f"{k:56s}{hb[k]:8d}{ha[k]:8d}{ha[k]-hb[k]:+8d}")

In [ ]:
# Side-by-side on one row the adapter fixed.
for i, (b, a) in enumerate(zip(baseline_records, tuned_records)):
    if (not b["passed"]) and a["passed"]:
        print("PROMPT (truncated):\n", b["row"]["prompt"][:400], "\n")
        print("=" * 46, "BEFORE", "=" * 46, "\n", b["completion"][:500])
        print("errors:", b["errors"][:2])
        print("=" * 47, "AFTER", "=" * 47, "\n", a["completion"][:500])
        break

In [ ]:
trainer.save_model(ADAPTER_DIR)          # LoRA adapter only (a few MB)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved:", sorted(os.listdir(ADAPTER_DIR)))

# Optional: fold the adapter into the base weights for a single deployable checkpoint.
# merged = model.merge_and_unload(); merged.save_pretrained("lfm25-350m-struct-merged")
#
# Optional: publish it.
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("your-username/lfm25-350m-ifstruct-grpo")

## 10. Serving it

Three rules for structured output in production:

1. **Decode cold.** `do_sample=False`. Sampling was for exploration during training; at
   serve time it only invents new ways to break a schema.
2. **Validate every response.** GRPO raises the pass rate; it does not make it 1.0. The
   validator is ~1 ms — always run it, and hand a *failing* parse back to the model with
   the concrete error rather than retrying blind.
3. **Constrained decoding is complementary, not redundant.** `outlines`, `xgrammar` or
   `llguidance` can force well-formed output — but they fight the policy when the policy
   wants to write prose, which costs latency and coherence. GRPO reduces how often the
   grammar has to intervene, so the two stack well. Add the constraint engine *after* the
   model has internalised the format, not instead of it.

In [ ]:
def structured_generate(user_prompt, spec, max_retries=1, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    """Generate, validate against the official checker, and repair once on failure."""
    messages = [{"role": "user", "content": user_prompt}]
    for attempt in range(max_retries + 1):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id, use_cache=True)
        completion = tokenizer.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True)

        verdict = score_one(spec, completion)
        if verdict.passed:
            return {"ok": True, "attempts": attempt + 1, "text": completion}
        if attempt == max_retries:
            return {"ok": False, "attempts": attempt + 1, "text": completion, "errors": verdict.errors}
        # Feed the concrete validator errors back in -- far better than "try again".
        messages += [{"role": "assistant", "content": completion},
                     {"role": "user", "content": "That response was rejected:\n- "
                      + "\n- ".join(verdict.errors[:5])
                      + "\nReturn only the corrected document."}]


probe = EVAL_ROWS[0]
result = structured_generate(probe["prompt"], probe)
print(f"passed={result['ok']}  attempts={result['attempts']}")
print(result["text"][:600])

## 11. Troubleshooting and what to do next

### When the run misbehaves

| Symptom | Cause | Fix |
|---|---|---|
| Reward curve dead flat | LoRA LR too low, or targets matched nothing | Raise LR to 5e-5; re-check `TARGET_MODULES` against the printed module names (`out_proj`, not `o_proj`) |
| Every group scores 0 | Prompts too hard for the base policy | The dense `reward_schema_fields` term already softens this; if it is still 0, shorten schemas or cut the item counts early in training |
| Reward rises, pass rate doesn't | Reward hacking, or a weight imbalance | Re-run the mutation table in §7; raise the weight on `reward_ifstruct_pass` |
| `ArrowInvalid` building the dataset | Heterogeneous `json_schema` / `top_level_count` | Store both as JSON strings (§6) |
| `KeyError` inside a reward | TRL dropped the metadata columns | `remove_unused_columns=False` |
| OOM during generation | Group too large | `NUM_GENERATIONS=2`, `PER_DEVICE_BS=2`, or `MAX_COMPLETION_LEN=384` |
| Loss is `nan` on a T4 | bf16 on Turing hardware | `bf16=False, fp16=True` (the config above detects this) |
| Completion length grows, reward flat | Length-biased objective | Confirm `loss_type="dapo"`; add an explicit length penalty |
| Generations are all identical | Temperature too low | Training needs `temperature≈0.9`; greedy belongs at inference only |

### Getting from ~30% toward the mid-40s

The published 44.9% came from more of everything, in roughly this order of value:

1. **More and better training data.** 500 synthetic rows is the binding constraint here,
   not the step count. Widen the entity library, deepen the nesting, and push harder on
   the escaping axis — strings carrying quotes, newlines and unicode are where small
   models break first.
2. **More steps: 250–500**, with `num_generations=8` for a lower-variance advantage
   estimate. Cost scales with `steps × generations`, so budget accordingly.
3. **Curriculum.** Train on 1–2 item, flat-schema rows first, then mix in deep nesting.
4. **Then tune.** `r=32`, LR sweep, `beta=0.02` if the policy starts degenerating.

Re-run the *whole* evaluation at each change. A 150-row subset has a standard error of
roughly ±3–4 points, so a 2-point "improvement" on that sample is noise — the paired sign
test in §9 exists for exactly this reason.

### When not to use this recipe

This trains **form, not substance**. IFStruct explicitly does not score whether the
content is correct or useful. LFM2.5-350M is a strong base for extraction, invoices, tool
arguments and config objects; it is the wrong base for knowledge-heavy or open-ended
generation, and no amount of GRPO on format rewards will change that. If your failures
are *"the JSON was valid but the values were wrong"*, you need a different reward — and
probably a bigger model.

## References

* Monigatti, Burtenshaw & Paniego — [*Fine-tuning a 350M Model for Better Structured Outputs in 100 GRPO Steps*](https://huggingface.co/blog/grpo-with-trl-ifstruct), Hugging Face, 3 September 2026
* Liquid AI — [*IFStruct: Measuring structured-output compliance*](https://www.liquid.ai/blog/ifstruct-v1.0)
* [`Liquid4All/ifstruct`](https://github.com/Liquid4All/ifstruct) — evaluator + frozen 2,000-row test set
* [`LiquidAI/ifstruct-v1.0`](https://huggingface.co/datasets/LiquidAI/ifstruct-v1.0) — the same test set on the Hub
* [`LiquidAI/LFM2.5-350M`](https://huggingface.co/LiquidAI/LFM2.5-350M) · [*LFM2.5-350M: No Size Left Behind*](https://www.liquid.ai/blog/lfm2-5-350m-no-size-left-behind)
* [TRL `GRPOTrainer` docs](https://huggingface.co/docs/trl/en/grpo_trainer) · [DeepSeekMath (GRPO)](https://arxiv.org/abs/2402.03300) · [DAPO](https://arxiv.org/abs/2503.14476)